In [8]:
!pip install -q datasets transformers evaluate scikit-learn sentence-transformers

import numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

ds = load_dataset("Yelp/yelp_review_full")
train = pd.DataFrame(ds["train"].shuffle(seed=42).select(range(20000)))
test = pd.DataFrame(ds["test"].shuffle(seed=42).select(range(5000)))
tr, val = train_test_split(train, test_size=0.1, random_state=42, stratify=train["label"])

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def to_ds(df):
    d = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    return d.map(lambda b: tokenizer(b["text"], truncation=True, max_length=256), batched=True)

tr_ds, val_ds, te_ds = to_ds(tr), to_ds(val), to_ds(test)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

def compute_metrics(eval_pred):
    logits, y = eval_pred
    p = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(y, p), "macro_f1": f1_score(y, p, average="macro")}

args = TrainingArguments(
    output_dir="distilbert-yelp", num_train_epochs=3,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=3e-5, eval_strategy="epoch", save_strategy="no",
    fp16=True, report_to="none", seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics)
trainer.train()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  299MB            

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.039445,0.964011,0.592000,0.574337
2,0.825334,0.916498,0.607500,0.605831
3,0.692191,0.959265,0.613500,0.612647


TrainOutput(global_step=1689, training_loss=0.8301612668802499, metrics={'train_runtime': 349.5264, 'train_samples_per_second': 154.495, 'train_steps_per_second': 4.832, 'total_flos': 3576811115520000.0, 'train_loss': 0.8301612668802499, 'epoch': 3.0})

In [9]:
out = trainer.predict(te_ds)
preds = np.argmax(out.predictions, axis=-1)
print("Test Accuracy:", accuracy_score(test["label"], preds))
print("Test Macro F1:", f1_score(test["label"], preds, average="macro"))

trainer.save_model("distilbert-yelp-stars")
tokenizer.save_pretrained("distilbert-yelp-stars")

Test Accuracy: 0.612
Test Macro F1: 0.6125672119911102


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert-yelp-stars/tokenizer_config.json',
 'distilbert-yelp-stars/tokenizer.json')

In [10]:
from sentence_transformers import SentenceTransformer

st = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
emb = st.encode(test["text"].tolist(), batch_size=128, normalize_embeddings=True)
np.save("review_embeddings.npy", emb.astype("float32"))

pd.DataFrame({"text": test["text"].str.slice(0, 600),
              "stars": test["label"] + 1}).to_csv("reviews.csv", index=False)
print(emb.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(5000, 384)


In [11]:
!zip -r distilbert-yelp-stars.zip distilbert-yelp-stars
from google.colab import files
files.download("distilbert-yelp-stars.zip")
files.download("review_embeddings.npy")
files.download("reviews.csv")

  adding: distilbert-yelp-stars/ (stored 0%)
  adding: distilbert-yelp-stars/config.json (deflated 54%)
  adding: distilbert-yelp-stars/tokenizer.json (deflated 71%)
  adding: distilbert-yelp-stars/model.safetensors (deflated 8%)
  adding: distilbert-yelp-stars/training_args.bin (deflated 54%)
  adding: distilbert-yelp-stars/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
out = trainer.predict(te_ds)
preds = np.argmax(out.predictions, axis=-1)
print("Test Accuracy:", accuracy_score(test["label"], preds))
print("Test Macro F1:", f1_score(test["label"], preds, average="macro"))


Test Accuracy: 0.612
Test Macro F1: 0.6125672119911102
